In [10]:
from langchain_groq import ChatGroq
from dotenv import find_dotenv , load_dotenv
from langchain_core.prompts import ChatPromptTemplate

import os

load_dotenv(find_dotenv(),override=True)

if os.environ['GROQ_API_KEY']:
    print('Grok Api Key found sucessfully')
else:
    print('Grok API key is missing ')

Grok Api Key found sucessfully


In [2]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

# **Pydantic LLM Schema**

In [3]:
from pydantic import BaseModel,Field

from typing import TypedDict,List

class llm_schema(BaseModel):
    tasks:List[str]=Field(...,description="A list of task to be performed by the worker ")

llm_with_schema=llm.with_structured_output(llm_schema)

# **Graph Schema**

In [4]:


class graph_schema(TypedDict):
        tasks:List[str]
        querry:str
        results:List[str]
        summary:str    

# **Orchestrator**

In [ ]:

def orchestrator_node(state:graph_schema)->graph_schema:
    user_query=state['querry']
    prompt=ChatPromptTemplate([
        ('system',"You are an Orchestrator that breaks down user querry into tasks for the worker"),
        ("human",f"User query : {user_query}. Please Generate one prompt per task for the worker nodes to complete , Return Tasks in List Format")
    ])
    chain=prompt | llm_with_schema
    response=chain.invoke({"querry":user_query})
    state['tasks']=response.tasks

    return state

In [6]:
# Worker Node for mini tasks
def exectue(querry:str):
    response=llm.invoke(f'Please execute this task {querry}')
    return response.content

In [9]:
# Worker Node Main that allows multi threading

from concurrent.futures import ThreadPoolExecutor

def worker_node(state:graph_schema)-> graph_schema:
    tasks=state['tasks']
    results=[]

    with ThreadPoolExecutor (max_workers=len(tasks)) as executor:
        results_futures=executor.map(exectue,tasks)
        for result in results_futures:
            results.append(result)
    
    state['results']=results
    return state

In [ ]:
# Collector Node

def collector_node(state:graph_schema)-> graph_schema:
    results=state['results']
    prompt=ChatPromptTemplate([
        ('system',"You are a collector that summerizes that results from the worker."),
        ('human',f"Here are the results from that worker: {results}. Summerize them in conise manner ")

    ])
